# CEPHA29-MD — stable fine-tune + crop refiner → md29c

Fine-tunes the app's 29-landmark model on **Aariz (all 29)** + **ISBI 2015 (only the 19
shared)** with per-sample visibility masks, then trains a **crop refiner** for the
landmarks with the largest errors. Background: `docs/AI_DIAGNOSIS_2026-11.md`.

What changed since md29b (whose training was unstable: NaN validation on 4 of 11
epochs, best checkpoint at epoch 3):

- **Stable training** (`scripts/ai/cephalohrnet_stability_patch.py`): FP32 validation,
  an EMA of the weights (validated and saved), checkpoint choice on the mean error with
  each point capped at 10 mm (one bad film can't decide it), and a per-epoch list of
  films that blow up (`val_outliers.jsonl`). Training stays fp16.
- **No inversion augmentation.** Mirror 0.5 and film-frame borders 0.3 stay; inverted
  films are handled by the app's polarity search (a brightness test can't replace it).
- **True pixel sizes in training validation** (Aariz `cephalogram_machine_mappings.csv`).
- **Crop refiner** (`scripts/ai/refiner.py`) for Po, PNS, Go, Ar, Or, ANS: a ResNet-18
  heatmap model on a 25.6 mm crop at 256 px (~0.1 mm/px, vs ~1 mm per heatmap pixel in
  the main model). Crops are sized from S–N, so no calibration is needed.
- **Evaluation** on the same held-out films as before, plus `refine.csv`
  (`scripts/ai/refine_eval.py`): each model alone, + refiner, + refiner gated, and the
  md29c + md29b ensemble.

**Kaggle setup**: GPU T4 x2, Internet On; attach `best_md29b.pt` (from the md29b
deliverables) and the Aariz dataset. ISBI comes from the GitHub mirror. Push the app
repo first: the notebook clones it from GitHub. About 6–7 h in total.

Outputs: `/kaggle/working/deliverables_md29c/` (`landmarks-md29c.*`,
`landmarks-refiner6.*`, `best_md29c.pt`, `refiner6.pt`, `audit/`).


In [ ]:
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy pandas requests seaborn huggingface_hub
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
# The mask patch edits CephaloHRNet source by exact match, so pin the commit it was
# validated against (an upstream change could otherwise break the patch mid-run).
CEPHALOHRNET_SHA = "a852d46b43c80796254588d3a8c8df57d30fe1ad"
!rm -rf CephaloHRNet CephalometryStudio
!git init -q CephaloHRNet
!git -C CephaloHRNet fetch -q --depth 1 https://github.com/Cestovatels/CephaloHRNet.git {CEPHALOHRNET_SHA}
!git -C CephaloHRNet checkout -q FETCH_HEAD
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.exists("CephaloHRNet/models/hrnet.py")
import subprocess
head = subprocess.run(["git", "-C", "CephaloHRNet", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == CEPHALOHRNET_SHA, head
assert os.path.exists("CephalometryStudio/scripts/ai/cephalohrnet_mask_patch.py"), "push the mask patch first"
assert os.path.exists("CephalometryStudio/scripts/ai/cephalohrnet_aug_patch.py"), "push the augmentation patch first"
for f in ("cephalohrnet_stability_patch.py", "refiner.py", "refine_eval.py"):
    assert os.path.exists("CephalometryStudio/scripts/ai/" + f), "push the app repo first: " + f
print("cloned")

In [ ]:
import glob, os, sys, urllib.request
import numpy as np
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from canonical import CEPHA29, ISBI19, ISBI_TO_CEPHA, isbi_group, isbi_masked

BASE = "https://raw.githubusercontent.com/stolariks/medical-landmark-detection/main/data/isbi-2015/"
ANN_TRAIN = "/kaggle/working/isbi_train.csv"
ANN_TEST = "/kaggle/working/isbi_test.csv"
for url, dest in [(BASE + "train/annotations.csv", ANN_TRAIN),
                  (BASE + "test/annotations.csv", ANN_TEST)]:
    if not os.path.exists(dest):
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
            f.write(r.read())

def read_isbi_csv(path):
    import csv
    out = {}
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if not row or row[0] == "image_path":
                continue
            try:
                pts = np.array([float(v) for v in row[1:]], np.float64).reshape(-1, 2)
            except Exception:
                continue
            if pts.shape[0] == len(ISBI19):
                out[os.path.splitext(os.path.basename(row[0]))[0]] = pts
    return out

isbi_gt = {"train": read_isbi_csv(ANN_TRAIN), "test": read_isbi_csv(ANN_TEST)}
print("ISBI GT:", {k: len(v) for k, v in isbi_gt.items()})

def index_input_images(stems):
    # stem -> path, per film, across every attached input. One folder may hold only
    # some films (e.g. an earlier run's eval29/isbi19 has just the 50 test films).
    found = {}
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        for p in glob.glob("/kaggle/input/**/" + ext, recursive=True):
            s = os.path.splitext(os.path.basename(p))[0]
            if s in stems and s not in found:
                found[s] = p
    return found

ISBI_INPUT = index_input_images(set(isbi_gt["train"]) | set(isbi_gt["test"]))
print("ISBI films found in /kaggle/input:", len(ISBI_INPUT), "(the rest come from the mirror)")

def isbi_image(split, stem):
    if stem in ISBI_INPUT:
        return ISBI_INPUT[stem]
    cache = "/kaggle/working/isbi_img_" + split
    os.makedirs(cache, exist_ok=True)
    for ext in (".jpg", ".png"):
        dest = os.path.join(cache, stem + ext)
        if os.path.exists(dest):
            return dest
        try:
            url = BASE + split + "/cepha400/" + stem + ext
            req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                f.write(r.read())
            return dest
        except Exception:
            continue
    return None

isbi_cases = {"train": [], "test": []}
for split in ("train", "test"):
    for stem, gt in isbi_gt[split].items():
        p = isbi_image(split, stem)
        if p:
            isbi_cases[split].append((stem, p, gt))
    print("ISBI", split, "cases:", len(isbi_cases[split]))

def find_dataset(root):
    # The Aariz root: train, valid AND test, each with Cephalograms and the
    # per-annotator landmark folders. Other attached datasets (checkpoints, earlier
    # md29 builds) can have a train/ folder too, so a looser test picks the wrong one.
    for base, dirs, _ in os.walk(root):
        ok = all(os.path.isdir(os.path.join(base, s, "Cephalograms")) and
                 os.path.isdir(os.path.join(base, s, "Annotations", "Cephalometric Landmarks"))
                 for s in ("train", "valid", "test"))
        if ok:
            return base
    return None

# Aariz, in order: an extracted copy among the inputs (a Kaggle dataset, or an earlier
# run's output, which keeps /kaggle/working/aariz), an Aariz.zip among the inputs,
# then figshare. Figshare now refuses automated downloads (HTTP 403), so in practice
# attach it: download Aariz.zip in a browser from https://doi.org/10.6084/m9.figshare.27986417
# and add it as a private Kaggle dataset.
import zipfile
AARIZ_ZIP_SIZE = 2098209792
AARIZ = find_dataset("/kaggle/input") or find_dataset("/kaggle/working/aariz")
if AARIZ is None:
    zips = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True)
            if os.path.basename(z).lower() == "aariz.zip" or os.path.getsize(z) == AARIZ_ZIP_SIZE]
    ZIP = zips[0] if zips else "/kaggle/working/Aariz.zip"
    if not zips and not (os.path.exists(ZIP) and os.path.getsize(ZIP) == AARIZ_ZIP_SIZE):
        try:
            req = urllib.request.Request("https://ndownloader.figshare.com/files/51041642",
                                         headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=900) as r, open(ZIP, "wb") as f:
                while True:
                    c = r.read(1 << 20)
                    if not c:
                        break
                    f.write(c)
        except Exception as e:
            print("figshare download failed:", e)
    if os.path.exists(ZIP) and zipfile.is_zipfile(ZIP):
        print("extracting", ZIP)
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/kaggle/working/aariz")
        AARIZ = find_dataset("/kaggle/working/aariz")
if AARIZ is None:
    cands = sorted({os.path.dirname(os.path.dirname(d)) for d in
                    glob.glob("/kaggle/input/**/Cephalograms", recursive=True)})
    print("Folders with Cephalograms under /kaggle/input:", *cands[:20], sep="\n  ")
print("AARIZ:", AARIZ)
assert AARIZ, ("Aariz not found. Download Aariz.zip in a browser from "
               "https://doi.org/10.6084/m9.figshare.27986417, add it to Kaggle as a private "
               "dataset (Datasets > New Dataset), attach it, and re-run this cell.")

In [ ]:
import glob, json, os, shutil
import numpy as np

MD = "/kaggle/working/md29"
EVAL = "/kaggle/working/eval29"
for d in (MD, EVAL):
    shutil.rmtree(d, ignore_errors=True)

def ann_dir(root):
    # CephaloHRNet resolves --annotators Merged29 under this directory.
    return os.path.join(root, "Annotations", "Cephalometric Landmarks", "Merged29")

def read_syms(path):
    data = json.load(open(path, encoding="utf-8"))
    return {lm["symbol"]: (float(lm["value"]["x"]), float(lm["value"]["y"]))
            for lm in data.get("landmarks", []) if "x" in (lm.get("value") or {})}

def aariz_avg(split, stem):
    root = os.path.join(AARIZ, split, "Annotations", "Cephalometric Landmarks")
    dirs = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    js = [os.path.join(d, stem + ".json") for d in dirs
          if os.path.exists(os.path.join(d, stem + ".json"))]
    if not js:
        return None
    sets = [read_syms(p) for p in js]
    keys = set(sets[0])
    for s in sets[1:]:
        keys &= set(s)
    return {k: (float(np.mean([s[k][0] for s in sets])),
                float(np.mean([s[k][1] for s in sets]))) for k in keys}

def write_case(root, prefix, domain, stem, img_path, pts29, visible=None):
    ceph = os.path.join(root, "Cephalograms")
    ad = ann_dir(root)
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    new_stem = (domain + "_" + stem) if prefix else stem
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"):
        ext = ".jpg"
    dst = os.path.join(ceph, new_stem + ext)
    if not os.path.exists(dst):
        shutil.copy(img_path, dst)
    doc = {"ceph_id": new_stem,
           "landmarks": [{"symbol": s, "value": {"x": pts29[s][0], "y": pts29[s][1]}} for s in CEPHA29]}
    if visible is not None:
        doc["visible"] = visible
    json.dump(doc, open(os.path.join(ad, new_stem + ".json"), "w"))

def aariz_pts(split, stem):
    avg = aariz_avg(split, stem)
    if not avg or not all(s in avg for s in CEPHA29):
        return None
    return {s: avg[s] for s in CEPHA29}

def write_eval19(root, stem, img_path, pts29):
    # Aariz case for 19-channel models: the 19 shared landmarks under ISBI symbols.
    ceph = os.path.join(root, "Cephalograms")
    ad = os.path.join(root, "Annotations", "Cephalometric Landmarks", "Merged19")
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    shutil.copy(img_path, os.path.join(ceph, stem + os.path.splitext(img_path)[1].lower()))
    doc = {"ceph_id": stem,
           "landmarks": [{"symbol": s, "value": {"x": pts29[ISBI_TO_CEPHA[s]][0], "y": pts29[ISBI_TO_CEPHA[s]][1]}}
                         for s in ISBI19]}
    json.dump(doc, open(os.path.join(ad, stem + ".json"), "w"))

def isbi_pts(gt, stem=None):
    # Map the 19 ISBI landmarks onto CEPHA29; the other 10 are masked sentinels.
    # With a stem, the landmarks this film labels in the other ISBI convention are
    # masked too (canonical.ISBI_MASK_BY_GROUP; docs/AI_DIAGNOSIS_2026-11.md), so the
    # model is taught one convention: Aariz's, which is the textbook one for Pog' and B.
    skip = {ISBI_TO_CEPHA[s] for s in isbi_masked(stem)} if stem is not None else set()
    mapped = {ISBI_TO_CEPHA[s]: (float(gt[i][0]), float(gt[i][1])) for i, s in enumerate(ISBI19)}
    pts = {s: mapped.get(s, (-1.0, -1.0)) for s in CEPHA29}
    visible = [s for s in CEPHA29 if s in mapped and s not in skip]
    return pts, visible

# Splits — the test sets stay out of training AND checkpoint selection:
#   train: Aariz train + ISBI train (minus 40 test, 40 valid)
#   valid: Aariz valid + 40 held-out ISBI train images (chooses best.pt)
#   eval:  Aariz test (29) + ISBI test (19 shared) — reported only. The mirror's ISBI
#          test films are all from 301-400, so 40 films from 1-300 are held out of
#          train as well: the benchmark scores both labelling conventions.
assert os.path.isdir(os.path.join(AARIZ, "test", "Cephalograms")), "Aariz test split not found"
ISBI_VAL, ISBI_TEST_A = 40, 40
isbi_train = sorted(isbi_cases["train"], key=lambda c: c[0])
group_a = [i for i, c in enumerate(isbi_train) if isbi_group(c[0]) == "1-300"]
test_a = set(np.random.default_rng(300).choice(group_a, size=ISBI_TEST_A, replace=False).tolist())
rest = [i for i in range(len(isbi_train)) if i not in test_a]
hold = set(np.random.default_rng(29).choice(rest, size=ISBI_VAL, replace=False).tolist())
isbi_split = {"train": [isbi_train[i] for i in rest if i not in hold],
              "valid": [isbi_train[i] for i in sorted(hold)]}
isbi_test = [isbi_train[i] for i in sorted(test_a)] + list(isbi_cases["test"])

for split in ("train", "valid"):
    for p in sorted(glob.glob(os.path.join(AARIZ, split, "Cephalograms", "*"))):
        stem = os.path.splitext(os.path.basename(p))[0]
        pts = aariz_pts(split, stem)
        if pts:
            write_case(os.path.join(MD, split), True, "aariz", stem, p, pts)
    for stem, p, gt in isbi_split[split]:
        pts, visible = isbi_pts(gt, stem)
        write_case(os.path.join(MD, split), True, "isbi", stem, p, pts, visible)

for p in sorted(glob.glob(os.path.join(AARIZ, "test", "Cephalograms", "*"))):
    stem = os.path.splitext(os.path.basename(p))[0]
    pts = aariz_pts("test", stem)
    if pts:
        write_case(os.path.join(EVAL, "aariz29"), False, "aariz", stem, p, pts)
        write_eval19(os.path.join(EVAL, "aariz19"), stem, p, pts)
import csv
for stem, p, gt in isbi_test:
    pts, visible = isbi_pts(gt)      # all 19 kept: the benchmark masks per film
    write_case(os.path.join(EVAL, "isbi19"), False, "isbi", stem, p, pts, visible)
with open(os.path.join(EVAL, "isbi19", "annotations.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image_path"] + [f"{k}_{a}" for k in range(1, 20) for a in ("x", "y")])
    for stem, p, gt in isbi_test:
        w.writerow([stem + os.path.splitext(p)[1].lower()] + [f"{v:g}" for v in np.ravel(gt)])
print("ISBI test films:", {g: sum(1 for c in isbi_test if isbi_group(c[0]) == g) for g in ("1-300", "301-400")})
n_masked = sum(len(isbi_masked(c[0])) for c in isbi_split["train"])
print(f"ISBI train: {len(isbi_split['train'])} films, {n_masked} labels masked for the other convention")

def count(root):
    return len(glob.glob(os.path.join(root, "Cephalograms", "*")))

def stems(root, strip_prefix):
    out = set()
    for p in glob.glob(os.path.join(root, "Cephalograms", "*")):
        s = os.path.splitext(os.path.basename(p))[0]
        out.add(s.split("_", 1)[1] if strip_prefix else s)
    return out

print("md29 train:", count(os.path.join(MD, "train")),
      "valid:", count(os.path.join(MD, "valid")),
      f"(ISBI {len(isbi_split['train'])}/{len(isbi_split['valid'])})",
      "| eval aariz29:", count(os.path.join(EVAL, "aariz29")),
      "aariz19:", count(os.path.join(EVAL, "aariz19")),
      "isbi19:", count(os.path.join(EVAL, "isbi19")))
assert count(os.path.join(MD, "train")) > 0 and count(os.path.join(MD, "valid")) > 0
assert count(os.path.join(EVAL, "aariz29")) > 0 and count(os.path.join(EVAL, "isbi19")) > 0
# No image may be both trained/selected on and reported.
seen = stems(os.path.join(MD, "train"), True) | stems(os.path.join(MD, "valid"), True)
reported = stems(os.path.join(EVAL, "aariz29"), False) | stems(os.path.join(EVAL, "isbi19"), False)
assert not (seen & reported), sorted(seen & reported)[:5]
print("no overlap between train/valid and the test sets")

# True pixel sizes, so validation (and best.pt) is in real mm: Aariz films from the
# dataset's machine mapping, ISBI 0.1 mm/px. CephaloHRNet reads <data>/cephalogram_machine_mappings.csv.
import csv
SPACING = next(iter(glob.glob(os.path.join(AARIZ, "**", "cephalogram_machine_mappings.csv"), recursive=True)), None)
print("Aariz pixel sizes:", SPACING or "NOT FOUND - Aariz mm assume 0.1 mm/px")
px = {}
if SPACING:
    with open(SPACING, newline="", encoding="utf-8", errors="replace") as f:
        rows_ = list(csv.reader(f))
    head = [h.strip().lower() for h in rows_[0]]
    idc = next((i for i, h in enumerate(head) if "id" in h), 0)
    pxc = next((i for i, h in enumerate(head) if "pixel" in h or "spacing" in h), len(head) - 1)
    for r_ in rows_[1:]:
        try:
            px["aariz_" + os.path.splitext(r_[idc].strip())[0]] = float(r_[pxc])
        except (ValueError, IndexError):
            pass
n_px = 0
with open(os.path.join(MD, "cephalogram_machine_mappings.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["cephalogram_id", "pixel_size"])
    for split in ("train", "valid"):
        for p_ in glob.glob(os.path.join(MD, split, "Cephalograms", "*")):
            sid = os.path.splitext(os.path.basename(p_))[0]
            v = px.get(sid, 0.1 if sid.startswith("isbi_") else None)
            if v:
                w.writerow([sid, v])
                n_px += 1
print("pixel sizes written for", n_px, "train/valid films")

In [ ]:
import sys
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from cephalohrnet_mask_patch import apply_mask_patches
from cephalohrnet_aug_patch import apply_aug_patches
from cephalohrnet_stability_patch import apply_stability_patches
apply_mask_patches("/kaggle/working/CephaloHRNet")
apply_aug_patches("/kaggle/working/CephaloHRNet")   # mirror / invert / film-frame flags
apply_stability_patches("/kaggle/working/CephaloHRNet")  # FP32 val, EMA, robust best.pt, outlier log
print("mask + augmentation + stability patches applied")


In [ ]:
import glob, os, shutil
import torch

# Preferred start: best_md29b.pt (the app's current model, in the md29b deliverables),
# then best_md29.pt. Either already knows all 29 landmarks and the convention masks, so
# a short fine-tune is enough. Fallback: the original 29-landmark best.pt, 80 epochs.
pref = {"best_md29b.pt": 0, "best_md29.pt": 1, "best.pt": 2, "best.pth": 3, "best.zip": 4}
cands = []
for name, rank in pref.items():
    for p in glob.glob("/kaggle/input/**/" + name, recursive=True):
        cands.append((rank, p))
cands.sort()
BEST = cands[0][1] if cands else None
if BEST and BEST.lower().endswith(".zip"):
    dest = "/kaggle/working/best.pt"
    if not os.path.exists(dest):
        shutil.copy(BEST, dest)
    BEST = dest
print("start checkpoint:", BEST)
assert BEST, "Attach best_md29b.pt (preferred) or the 29-landmark best.pt as a Kaggle dataset"
ck = torch.load(BEST, map_location="cpu", weights_only=False)
n_head = ck["model"]["head.weight"].shape[0]
print("checkpoint head channels:", n_head, "| epoch:", ck.get("epoch"))
assert n_head == 29, "expected a 29-landmark checkpoint"
FROM_FT = os.path.basename(BEST) in ("best_md29.pt", "best_md29b.pt")
# EMA + capped-error selection make the curve smoother, so patience can be a bit longer.
SCHEDULE = (dict(epochs=30, lr="2e-4", patience=10, warmup=1) if FROM_FT
            else dict(epochs=80, lr="5e-4", patience=15, warmup=5))
print("schedule:", "short fine-tune" if FROM_FT else "full run from the original model", SCHEDULE)


In [ ]:
import glob, os, shutil
import numpy as np
# Static INT8 calibration on Aariz films only. ISBI films have a dark frame whose top
# edge drives early activations ~100x above anatomy; MinMax (and Percentile) then set
# every layer's range from it and the INT8 model is degenerate (the 2026-11 Kaggle
# export; docs/AI_DIAGNOSIS_2026-11.md). Random, not sorted, so several scanners are in.
calib = "/kaggle/working/calib_md29"
shutil.rmtree(calib, ignore_errors=True)
os.makedirs(calib, exist_ok=True)
rng = np.random.default_rng(7)
files = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "aariz_*")))
for i in sorted(rng.choice(len(files), size=min(32, len(files)), replace=False).tolist()):
    shutil.copy(files[i], os.path.join(calib, os.path.basename(files[i])))
print("calib:", len(os.listdir(calib)), "Aariz films")


In [ ]:
import glob, os, subprocess, sys

for split in ("train", "valid"):
    n_img = len(glob.glob(os.path.join(MD, split, "Cephalograms", "*")))
    n_ann = len(glob.glob(os.path.join(MD, split, "Annotations", "Cephalometric Landmarks", "Merged29", "*.json")))
    print(f"{split}: images={n_img} annotations={n_ann}")
    assert n_img and n_ann, f"{split} split is empty"

# Augmentation: mirror 0.5 and film-frame borders 0.3. No inversion: md29b still failed
# on inverted films after it, and it shifts the batch-norm statistics; the app's polarity
# search handles inverted films (docs/AI_DIAGNOSIS_2026-11.md, "md29b results").
# Stability: training in fp16 (--amp), validation in FP32 (stability patch), EMA of the
# weights, best.pt on the mean error capped at 10 mm per point.
cmd = [sys.executable, "train.py",
       "--data", MD, "--model", "hrnet_w32",
       "--imgsz", "768", "--batch", "4", "--epochs", str(SCHEDULE["epochs"]),
       "--lr", SCHEDULE["lr"], "--warmup-epochs", str(SCHEDULE["warmup"]),
       "--flip-prob", "0.5", "--invert-prob", "0", "--border-prob", "0.3",
       "--ema-decay", "0.998", "--select", "clipped", "--clip-mm", "10", "--outlier-mm", "5",
       "--loss", "awing", "--amp", "--device", "0",
       "--workers", "0", "--patience", str(SCHEDULE["patience"]),
       "--project", "runs/train", "--name", "md29_w32",
       "--annotators", "Merged29", "--img-dir", "Cephalograms",
       "--pretrained-path", BEST]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd="/kaggle/working/CephaloHRNet", capture_output=True, text=True)
open("/kaggle/working/train_md29c.log", "w").write(r.stdout + "\n" + r.stderr)
keep = ("[", "  [BEST]", "  non-finite", "  Loaded", "Early stopping", "  Best val")
print("\n".join(l for l in r.stdout.splitlines()
                if l.startswith(keep) or "validation films above" in l)[-8000:])
print(r.stderr[-3000:])
r.check_returncode()
# Films that blew up, per epoch (empty lists are good news).
import json
runs = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/md29_*/val_outliers.jsonl"), key=os.path.getmtime)
for line in open(runs[-1]):
    e = json.loads(line)
    print(f"epoch {e['epoch']:2d}: bad steps {e['bad_steps']}, non-finite {len(e['nonfinite'])}, "
          f"over 5 mm {len(e['over'])}, median {e['median_mm']:.3f}, capped mean {e['clipped_mm']:.3f}")

In [ ]:
import glob, os, subprocess, sys
BESTR = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/md29_*/best.pt"),
               key=os.path.getmtime)[-1]
SAMPLE = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[0]
print("weights:", BESTR)
subprocess.run([sys.executable, "CephalometryStudio/scripts/export-landmark-onnx.py",
                "--repo", "/kaggle/working/CephaloHRNet",
                "--weights", BESTR, "--model", "hrnet_w32",
                "--num-landmarks", "29", "--input-size", "768",
                "--out", "/kaggle/working/landmarks-md29c.onnx",
                "--quantize", "--calib-dir", "/kaggle/working/calib_md29",
                "--verify", SAMPLE],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "CephalometryStudio/scripts/build-shape-model.py",
                "--set", "cepha29",
                "--ann-dir", os.path.join(AARIZ, "train", "Annotations", "Cephalometric Landmarks", "Senior Orthodontists"),
                "--k", "10",
                "--out", "/kaggle/working/shapeModel.cepha29.json"],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
# Crop refiner for the weak landmarks (scripts/ai/refiner.py). Same train/valid films
# as the main model; the test sets stay out. ~1 h on a T4.
REF_DIR = "/kaggle/working/refiner6"
REFINER = "/kaggle/working/landmarks-refiner6.onnx"
cmd = [sys.executable, "CephalometryStudio/scripts/ai/refiner.py", "train",
       "--data", MD, "--out", REF_DIR,
       "--landmarks", "Po", "PNS", "Go", "Ar", "Or", "ANS",
       "--size", "256", "--crop-mm", "25.6", "--epochs", "40", "--batch", "32",
       "--lr", "1e-3", "--patience", "10", "--workers", "2", "--pretrained", "--device", "cuda"]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd="/kaggle/working", capture_output=True, text=True)
open("/kaggle/working/train_refiner6.log", "w").write(r.stdout + "\n" + r.stderr)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
r.check_returncode()
subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/refiner.py", "export",
                "--weights", os.path.join(REF_DIR, "best.pt"), "--out", REFINER],
               cwd="/kaggle/working", check=True)


In [ ]:
import glob, hashlib, json, os, subprocess, sys, urllib.request
OUT = "/kaggle/working/audit_md29c"
os.makedirs(OUT, exist_ok=True)

# Reference models from the app's Hugging Face repo (SHA-256 as in landmarkModelInfo.js).
HF = "https://huggingface.co/MuhammadNabeelSh/cephalometry-landmarks/resolve/main/"
REFS = {
    "landmarks-md29b.int8.onnx": "d4f2096739f2cf9d40269ee67eebc5e49f7b159f8f4f06ac421036b951275c39",
}
for name, digest in REFS.items():
    dest = "/kaggle/working/ref_" + name
    if not os.path.exists(dest):
        urllib.request.urlretrieve(HF + name, dest)
    got = hashlib.sha256(open(dest, "rb").read()).hexdigest()
    assert got == digest, (name, got)

MODELS = [  # (tag, onnx, vocabulary)
    ("md29c_int8", "/kaggle/working/landmarks-md29c.int8.onnx", "cepha29"),
    ("md29c_fp32", "/kaggle/working/landmarks-md29c.onnx", "cepha29"),
    ("md29b_int8 (app now)", "/kaggle/working/ref_landmarks-md29b.int8.onnx", "cepha29"),
]
SPACING_ARGS = ["--spacing-csv", SPACING] if SPACING else []
ERR = []
for tag, model, vocab in MODELS:
    key = tag.split(" ")[0]
    os.makedirs(os.path.join(OUT, key), exist_ok=True)
    # External: ISBI test (19 shared), raw, production search.
    r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/external_benchmark.py",
                        "--model", model,
                        "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                        "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                        "--search", "--no-ssm", "--mask-conventions",
                        "--out", os.path.join(OUT, key, "isbi19")],
                       cwd="/kaggle/working")
    if r.returncode: ERR.append(key + " isbi")
    # In-domain: Aariz test, raw, all 29 landmarks.
    sub = "aariz29" if vocab == "cepha29" else "aariz19"
    ann = "Merged29" if vocab == "cepha29" else "Merged19"
    r = subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                        "--model", model,
                        "--set", vocab, "--limit", "200", "--dark",
                        "--ann-dir", os.path.join(EVAL, sub, "Annotations", "Cephalometric Landmarks", ann),
                        "--images", os.path.join(EVAL, sub, "Cephalograms"),
                        *SPACING_ARGS,
                        "--metrics-out", os.path.join(OUT, key, "aariz_metrics.json")],
                       cwd="/kaggle/working")
    if r.returncode: ERR.append(key + " aariz")
# Robustness / uncertainty / shape prior / quality gate (scripts/ai/robustness_eval.py):
# the INT8s only (what ships), new vs the app's current model. ~30 min each on CPU.
for tag, model, vocab in MODELS:
    key = tag.split(" ")[0]
    if not key.endswith("int8") or key.startswith("cepha29"):
        continue
    r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/robustness_eval.py",
                        "--model", model,
                        "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                        "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                        "--aariz-images", os.path.join(EVAL, "aariz29", "Cephalograms"),
                        "--aariz-ann", os.path.join(EVAL, "aariz29", "Annotations", "Cephalometric Landmarks", "Merged29"),
                        *(["--aariz-spacing", SPACING] if SPACING else []),
                        "--aariz-limit", "100",
                        "--ssm", "/kaggle/working/shapeModel.cepha29.json",
                        "--out", os.path.join(OUT, key, "robustness")],
                       cwd="/kaggle/working", capture_output=True, text=True)
    print(key, "robustness:", "ok" if r.returncode == 0 else r.stderr[-2000:])
    if r.returncode: ERR.append(key + " robustness")
# Crop refiner + ensemble on the same held-out films (all Aariz test, all ISBI test).
r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/refine_eval.py",
                    "--model", "md29c_int8=/kaggle/working/landmarks-md29c.int8.onnx",
                    "--model", "md29b_int8=/kaggle/working/ref_landmarks-md29b.int8.onnx",
                    "--refiner", REFINER, "--gate-mm", "4",
                    "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                    "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                    "--aariz-images", os.path.join(EVAL, "aariz29", "Cephalograms"),
                    "--aariz-ann", os.path.join(EVAL, "aariz29", "Annotations", "Cephalometric Landmarks", "Merged29"),
                    *(["--aariz-spacing", SPACING] if SPACING else []),
                    "--out", os.path.join(OUT, "refine")],
                   cwd="/kaggle/working", capture_output=True, text=True)
print("refine eval:", "ok" if r.returncode == 0 else r.stderr[-2000:])
if r.returncode: ERR.append("refine")
print("failed:", ERR)

In [ ]:
import json, os
import numpy as np
import pandas as pd

SHARED = [ISBI_TO_CEPHA[s] for s in ISBI19]           # CEPHA29 symbols of the shared 19
EXTRA = [s for s in CEPHA29 if s not in SHARED]       # the 10 Aariz-only landmarks
TO_ISBI = {c: s for s, c in ISBI_TO_CEPHA.items()}

def load(path):
    try:
        return json.load(open(path))
    except Exception:
        return None

def lm_mre(per, syms, rename=None):
    vals = [per.get((rename or {}).get(s, s), {}).get("mre_mm") for s in syms]
    vals = [v for v in vals if isinstance(v, (int, float)) and np.isfinite(v)]
    return round(float(np.mean(vals)), 2) if len(vals) == len(syms) else None

rows = []
for tag, _, vocab in MODELS:
    key = tag.split(" ")[0]
    isbi = load(os.path.join(OUT, key, "isbi19", "isbi_metrics.json")) or {}
    aar = load(os.path.join(OUT, key, "aariz_metrics.json")) or {}
    io, ao, per = isbi.get("overall", {}), aar.get("overall", {}), aar.get("per_landmark", {})
    rename = None
    rows.append({
        "model": tag,
        "ISBI MRE": io.get("mre_mm"), "ISBI median": io.get("median_mm"), "ISBI SDR@2": io.get("sdr_2.0mm"),
        "ISBI 1-300 MRE": isbi.get("by_group", {}).get("1-300", {}).get("mre_mm"),
        "ISBI 301-400 MRE": isbi.get("by_group", {}).get("301-400", {}).get("mre_mm"),
        "Aariz shared-19 MRE": lm_mre(per, SHARED, rename),
        "Aariz extra-10 MRE": lm_mre(per, EXTRA) if vocab == "cepha29" else None,
        "Aariz all MRE": ao.get("mre_mm"), "Aariz median": ao.get("median_mm"), "Aariz SDR@2": ao.get("sdr_2.0mm"),
    })
table = pd.DataFrame(rows).set_index("model").astype(float).round(2)
table.to_csv(os.path.join(OUT, "comparison.csv"))

# Refiner and ensemble (scripts/ai/refine_eval.py): overall, per group, and the weak 6.
refine = pd.read_csv(os.path.join(OUT, "refine", "refine.csv")).set_index("variant") \
    if os.path.exists(os.path.join(OUT, "refine", "refine.csv")) else pd.DataFrame()
if len(refine):
    print("\n=== refiner / ensemble (mm) ===")
    print(refine.round(2).to_string())
    rj = load(os.path.join(OUT, "refine", "refine.json")) or {}
    print("refiner shift vs error change (negative = better):", rj.get("shift_vs_gain"))

# Robustness (INT8s): one pass per orientation, uncertainty, shape prior, gate suggestion.
rob = []
for tag, _, _ in MODELS:
    key = tag.split(" ")[0]
    r = load(os.path.join(OUT, key, "robustness", "robustness.json"))
    if not r:
        continue
    o = r["orientation"]
    def m(c, d):
        return o.get(c, {}).get(d, {}).get("mre_mm")
    rob.append({
        "model": tag,
        **{f"{d} {c} MRE": m(c, d) for d in ("isbi", "aariz") for c in ("normal", "mirrored", "inverted", "mirrored+inverted")},
        "flip-TTA MRE": r["flip_tta"]["mre_mm"],
        "SSM always MRE": (r.get("ssm") or {}).get("always", {}).get("mre_mm"),
        "rho flip-dist vs err": r["uncertainty"]["spearman_flip_distance_vs_error"],
        "rho PSR vs err": r["uncertainty"]["spearman_psr_vs_error"],
        "gate residualRatioMedium": (r["quality"].get("suggested") or {}).get("residualRatioMedium"),
        "gate residualRatioHigh": (r["quality"].get("suggested") or {}).get("residualRatioHigh"),
        "gate separable": (r["quality"].get("suggested") or {}).get("separable"),
    })
robust = pd.DataFrame(rob).set_index("model") if rob else pd.DataFrame()
robust.to_csv(os.path.join(OUT, "robustness.csv"))
print(robust.T.to_string())
print("Held-out test sets: ISBI test (19 shared; 40 films from 1-300 + the mirror's test films")
print("from 301-400, each scored without the landmarks it labels in the other convention) and Aariz test.")
print("md29b (app now) was fine-tuned from md29, which saw ISBI films 1-300 in training, so its")
print("'ISBI 1-300' column is not fully held out; md29c starts from md29b, so the same applies.")
print("Aariz mm use each film's true pixel size when cephalogram_machine_mappings.csv was found.")
table

In [ ]:
import glob, hashlib, os, shutil
out = "/kaggle/working/deliverables_md29c"
os.makedirs(out, exist_ok=True)
for name in ("landmarks-md29c.onnx", "landmarks-md29c.int8.onnx", "landmarks-md29c-verify.png",
             "landmarks-refiner6.onnx", "landmarks-refiner6.json", "shapeModel.cepha29.json",
             "train_md29c.log", "train_refiner6.log"):
    for p in glob.glob("/kaggle/working/" + name):
        shutil.copy(p, out)
# Trained checkpoints: the starting points for any later fine-tune (e.g. more landmarks).
shutil.copy(BESTR, os.path.join(out, "best_md29c.pt"))
shutil.copy(os.path.join(REF_DIR, "best.pt"), os.path.join(out, "refiner6.pt"))
run_dir = os.path.dirname(BESTR)
for name in ("val_outliers.jsonl", "args.json", "metrics.csv", "results.csv"):
    if os.path.exists(os.path.join(run_dir, name)):
        shutil.copy(os.path.join(run_dir, name), os.path.join(out, "train_" + name))
if os.path.exists(os.path.join(REF_DIR, "log.json")):
    shutil.copy(os.path.join(REF_DIR, "log.json"), os.path.join(out, "refiner6_log.json"))
for p in glob.glob(OUT + "/**/*", recursive=True):
    if os.path.isfile(p):
        rel = os.path.relpath(p, OUT)
        dst = os.path.join(out, "audit", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

print("=== DELIVERABLES ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {sha(p)}")


In [ ]:
# Optional: copy the deliverables to a PRIVATE Hugging Face repo, so they outlive
# this Kaggle session. Needs a write token as the Kaggle secret HF_TOKEN
# (Add-ons → Secrets); without it this cell only prints a note.
import datetime, os
from huggingface_hub import HfApi

HF_REPO = "MuhammadNabeelSh/cephalometry-training"   # private; separate from the public model repo

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = hf_token()
if not token:
    print("HF_TOKEN not set — skipped the Hugging Face upload. Download deliverables_md29c/ from Output.")
else:
    try:
        api = HfApi(token=token)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        dest = "md29c/" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M")
        api.upload_folder(repo_id=HF_REPO, repo_type="model", folder_path=out,
                          path_in_repo=dest, commit_message="md29c deliverables " + dest)
        print(f"uploaded to https://huggingface.co/{HF_REPO}/tree/main/{dest}")
    except Exception as e:
        # The run's results are still in Output; an upload failure must not fail the run.
        print("Hugging Face upload failed:", e)

## Next

1. Download `/kaggle/working/deliverables_md29c/` (keep `best_md29c.pt` and `refiner6.pt`).
   With the `HF_TOKEN` secret set, they are also in the private repo
   `MuhammadNabeelSh/cephalometry-training` under `md29c/<date-time>/`.
2. Check training first: the epoch lines in cell 8's output (or `train_val_outliers.jsonl`).
   Non-finite counts should be 0 and the capped mean should fall smoothly.
3. `audit/comparison.csv`: md29c vs md29b (app now) on ISBI 1-300 / 301-400 and Aariz.
4. `audit/refine/refine.csv`: does `+refiner` (or `+refiner gated`) lower Po, PNS, Go, Ar,
   Or, ANS without raising the overall MRE? Is the ensemble worth a second pass?
   `refine.json` → `shift_vs_gain` shows which refiner moves help (negative = better).
5. `audit/robustness.csv`: mirrored-film accuracy (the orientation search stays on either
   way), flip consistency vs PSR, gate thresholds.
6. If md29c wins: host `landmarks-md29c.int8.onnx` and switch `LANDMARK_MODEL`. If the
   refiner helps: host `landmarks-refiner6.onnx` + `.json`; the app needs the crop step
   (same geometry as `scripts/ai/refiner.py`) before it can use it.
